# Week 6 - Day 1
## Introducción a Deep Learning vs Machine Learning tradicional + ANNs + Polynomial Fitting

**Autor:** Alex Goldbaum  
**Curso:** Developers Institute - Bootcamp Data Science

Ejercicios:
1. Tabla comparativa ML tradicional vs Deep Learning
2. Diagrama de una ANN simple
3. Generación y visualización de dataset con ruido
4. Ajuste de modelos polinómicos de distintos grados (overfitting)
5. Cross-validation para encontrar el grado óptimo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

---
## 🌟 Ejercicio 1: Deep Learning vs Traditional Machine Learning

### Tabla comparativa

| Aspecto | Machine Learning tradicional | Deep Learning |
|---|---|---|
| **Feature Engineering** | Manual, requiere experto del dominio | Automática, la red aprende las features |
| **Data Processing** | Funciona bien con datos estructurados/tabulares | Brilla con datos no estructurados (imágenes, texto, audio) |
| **Scalability** | El rendimiento se estanca con mucho dato | Mejora a medida que crece el dataset |
| **Pattern Discovery** | Patrones lineales o poco profundos | Patrones jerárquicos y no lineales muy complejos |
| **Computational Requirements** | CPU suele bastar, entrenamiento rápido | Requiere GPU/TPU, entrenamiento costoso |

### Casos de uso

**Problema mejor para ML tradicional:** Predicción de *churn* (abandono) de clientes con datos tabulares de pocas columnas y miles —no millones— de registros. Un Random Forest o una regresión logística rinden igual o mejor que una red neuronal, son interpretables y más baratos de entrenar.

**Problema mejor para Deep Learning:** Diagnóstico médico a partir de imágenes (por ejemplo, detección de neumonía en radiografías). Las CNN aprenden representaciones jerárquicas de píxeles que serían imposibles de *feature-engineerar* a mano.

### ¿Por qué Deep Learning gana en datos no estructurados?

Deep Learning destaca con datos no estructurados porque sus capas aprenden representaciones jerárquicas de forma automática: las primeras capas detectan patrones simples (bordes, fonemas, n-gramas) y las capas profundas combinan esos patrones en conceptos abstractos (caras, objetos, intención semántica). En ML tradicional ese paso requiere feature engineering manual que es lento, frágil y limitado por el conocimiento del experto. Además, las redes profundas escalan con los datos: a más imágenes/texto/audio mejor representación aprenden, mientras que los modelos clásicos se saturan rápido. Por último, arquitecturas como CNN, RNN y Transformers incorporan *inductive biases* (convolución, recurrencia, atención) que se alinean naturalmente con la estructura de los datos crudos.

---
## 🌟 Ejercicio 2: Artificial Neural Networks (ANNs)

### Diagrama de la red

```
        Input layer        Hidden layer          Output layer
        (3 neuronas)       (4 neuronas)          (2 neuronas)

           x1 ●----w11---->●----.
                  \        ●----.\
           x2 ●----w22---->●----. \--> ● y1
                  \        ●----. /
           x3 ●----w33---->●----./--> ● y2

                  ↑           ↑              ↑
               pesos (w)   activación f()  salida
```

### Componentes etiquetados

- **Neuronas (●):** unidades de cómputo (entradas, ocultas y salidas).
- **Pesos (w_ij):** fuerza de la conexión entre neuronas de capas consecutivas.
- **Bias (b):** término aditivo por neurona que desplaza la activación.
- **Función de activación f():** no linealidad aplicada a `(Σ w·x + b)`, por ejemplo ReLU, sigmoide o tanh.
- **Capas:** input → hidden → output.

### Flujo de información (forward pass)

1. Cada neurona oculta calcula `z = Σ (w_ij · x_i) + b_j`.
2. Se aplica la función de activación `a = f(z)`, introduciendo no linealidad.
3. Las activaciones de la capa oculta se propagan a la capa de salida usando otra matriz de pesos y bias.
4. La capa de salida produce las predicciones (por ejemplo softmax para clasificación). Durante el entrenamiento, la pérdida se propaga hacia atrás (*backpropagation*) para ajustar pesos y biases con descenso de gradiente.

In [ ]:
# Diagrama opcional de la ANN con matplotlib
fig, ax = plt.subplots(figsize=(9, 6))

layers = {'Input': (1, [1, 2, 3]),
          'Hidden': (3, [0.5, 1.5, 2.5, 3.5]),
          'Output': (5, [1.5, 2.5])}

colors = {'Input': 'steelblue', 'Hidden': 'orange', 'Output': 'crimson'}

positions = {}
for name, (x, ys) in layers.items():
    for i, y in enumerate(ys):
        positions[(name, i)] = (x, y)
        ax.scatter(x, y, s=900, color=colors[name], zorder=3, edgecolors='black')
        ax.text(x, y, f'{name[0]}{i+1}', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
    ax.text(x, max(ys) + 0.7, f'{name} layer', ha='center', fontsize=12, fontweight='bold')

# conexiones input -> hidden
for i in range(3):
    for j in range(4):
        ax.plot([1, 3], [layers['Input'][1][i], layers['Hidden'][1][j]], 'gray', alpha=0.4, zorder=1)
# conexiones hidden -> output
for j in range(4):
    for k in range(2):
        ax.plot([3, 5], [layers['Hidden'][1][j], layers['Output'][1][k]], 'gray', alpha=0.4, zorder=1)

ax.set_xlim(0, 6)
ax.set_ylim(-0.5, 5)
ax.axis('off')
ax.set_title('ANN: 3 inputs → 4 hidden → 2 outputs', fontsize=13)
plt.tight_layout()
plt.show()

---
## 🌟 Ejercicio 3: Creación del dataset y visualización

Generamos 20 puntos con `y = -x²` + ruido gaussiano `N(0, 0.05)`, los visualizamos y separamos en train (12) y test (8).

In [ ]:
np.random.seed(0)
x = np.arange(-1, 1, 0.1)
y = -x**2 + np.random.normal(0, 0.05, len(x))

x_train, y_train = x[:12], y[:12]
x_test, y_test = x[12:], y[12:]

print(f'Total puntos: {len(x)}')
print(f'Train: {len(x_train)} puntos')
print(f'Test:  {len(x_test)} puntos')

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(x_train, y_train, color='steelblue', label='Train (primeros 12)')
plt.scatter(x_test, y_test, color='orangered', label='Test (últimos 8)')
plt.title('Dataset: y = -x² + ruido N(0, 0.05)')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## 🌟 Ejercicio 4: Ajuste de modelos polinómicos

Definimos `polynomial_fit(degree)` y `plot_polyfit(degree)` y visualizamos los ajustes para grados 1, 7 y 11.

In [ ]:
def polynomial_fit(degree):
    """Devuelve un np.poly1d ajustado a los datos de entrenamiento."""
    coefs = np.polyfit(x_train, y_train, degree)
    return np.poly1d(coefs)


def plot_polyfit(degree):
    """Plotea train, test y la curva polinómica ajustada."""
    poly = polynomial_fit(degree)
    x_curve = np.linspace(-1, 1, 200)

    plt.figure(figsize=(7, 5))
    plt.scatter(x_train, y_train, color='steelblue', label='Train')
    plt.scatter(x_test, y_test, color='orangered', label='Test')
    plt.plot(x_curve, poly(x_curve), color='black', label=f'Polinomio grado {degree}')
    plt.title(f'Ajuste polinómico de grado {degree}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.ylim(-1.5, 0.5)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
plot_polyfit(1)

In [ ]:
plot_polyfit(7)

In [ ]:
plot_polyfit(11)

### Observaciones

- **Grado 1:** underfitting evidente. Una recta no captura la curvatura de `-x²`.
- **Grado 7:** ajuste cercano a los datos de train, todavía razonable en test.
- **Grado 11:** overfitting fuerte. La curva pasa exacto por los puntos de train pero diverge en los extremos y falla en test.

---
## 🌟 Ejercicio 5: Cross-validation para encontrar el grado óptimo

Recorremos grados 1 a 11, calculamos RMSE de train y test, y graficamos en escala log.

In [ ]:
results = []
for degree in range(1, 12):
    poly = polynomial_fit(degree)
    rmse_train = np.sqrt(mean_squared_error(y_train, poly(x_train)))
    rmse_test = np.sqrt(mean_squared_error(y_test, poly(x_test)))
    results.append((degree, rmse_train, rmse_test))

print(f"{'Degree':>6} | {'RMSE Train':>12} | {'RMSE Test':>12}")
print('-' * 38)
for d, rt, rv in results:
    print(f'{d:>6} | {rt:>12.5f} | {rv:>12.5f}')

best = min(results, key=lambda r: r[2])
print(f'\nGrado óptimo (mín. RMSE test): {best[0]}')
print('Coincide con el modelo verdadero y = -x² (grado 2).')

In [ ]:
degrees = [r[0] for r in results]
rmse_train = [r[1] for r in results]
rmse_test = [r[2] for r in results]

plt.figure(figsize=(8, 5))
plt.plot(degrees, rmse_train, 'o-', label='RMSE Train', color='steelblue')
plt.plot(degrees, rmse_test, 's-', label='RMSE Test', color='orangered')
plt.axvline(best[0], color='green', linestyle='--', alpha=0.6, label=f'Grado óptimo = {best[0]}')
plt.yscale('log')
plt.xlabel('Grado del polinomio')
plt.ylabel('RMSE (escala log)')
plt.title('RMSE vs grado del polinomio')
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.show()

### Conclusión

- El RMSE de **train** baja monótonamente al subir el grado: más flexibilidad → mejor ajuste a los puntos vistos.
- El RMSE de **test** baja hasta el grado 2 (el verdadero) y después **sube** porque el modelo memoriza ruido.
- Cross-validation confirma que el grado 2 es el óptimo, lo que recupera la estructura real `y = -x²`. Este es el clásico bias-variance tradeoff: pasado cierto punto, más complejidad empeora la generalización.